In [4]:
import pandas as pd

# Read only the columns we need
columns_needed = [
    'Rndrng_NPI', 'Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_Type',
    'Rndrng_Prvdr_State_Abrvtn', 'Tot_Mdcr_Pymt_Amt', 'Tot_Sbmtd_Chrg',
    'Tot_Mdcr_Alowd_Amt', 'Tot_Benes', 'Tot_Srvcs',
    'Bene_CC_PH_Diabetes_V2_Pct', 'Bene_CC_PH_Hypertension_V2_Pct',
    'Bene_CC_PH_HF_NonIHD_V2_Pct', 'Bene_CC_PH_CKD_V2_Pct'
]

df = pd.read_csv(r"C:\Docs\AWS Certification\cms-claims-analytics-pipeline\data\Medicare Physician & Other Practitioners - by Provider\2023\MUP_PHY_R25_P05_V20_D23_Prov.csv",
    usecols=columns_needed)

print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
df.head()

Rows: 1259343, Columns: 13


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,Tot_Benes,Tot_Srvcs,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,Tot_Mdcr_Pymt_Amt,Bene_CC_PH_CKD_V2_Pct,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct
0,1003000126,Enkeshafi,MD,Hospitalist,344,814.0,173087.77,78590.79,62198.36,50.0,45.0,53.0,75.0
1,1003000134,Cibull,IL,Pathology,2093,4839.0,726858.00,171256.75,127984.95,20.0,22.0,13.0,63.0
2,1003000142,Khalil,OH,Anesthesiology,325,1455.0,451425.00,155879.22,118122.12,28.0,44.0,24.0,75.0
3,1003000423,Velotta,OH,Obstetrics & Gynecology,63,119.0,13785.00,6388.39,5230.52,NaN,22.0,NaN,59.0
4,1003000480,Rothchild,CO,General Surgery,99,124.0,98271.00,20288.98,15794.97,27.0,33.0,16.0,74.0


In [5]:
df.isnull().sum()

Rndrng_NPI                             0
Rndrng_Prvdr_Last_Org_Name             0
Rndrng_Prvdr_State_Abrvtn              0
Rndrng_Prvdr_Type                      0
Tot_Benes                              0
Tot_Srvcs                              0
Tot_Sbmtd_Chrg                         0
Tot_Mdcr_Alowd_Amt                     0
Tot_Mdcr_Pymt_Amt                      0
Bene_CC_PH_CKD_V2_Pct             269638
Bene_CC_PH_Diabetes_V2_Pct        207938
Bene_CC_PH_HF_NonIHD_V2_Pct       356814
Bene_CC_PH_Hypertension_V2_Pct     56940
dtype: int64

In [6]:
df = df.fillna(0)

In [7]:
df[['Tot_Mdcr_Pymt_Amt', 'Tot_Benes', 'Tot_Srvcs']].describe()

,Tot_Mdcr_Pymt_Amt,Tot_Benes,Tot_Srvcs
count,1.259343e+06,1.259343e+06,1.259343e+06
mean,8.961897e+04,3.003412e+02,2.665236e+03
std,7.268204e+05,3.002691e+03,4.703013e+04
min,0.000000e+00,1.100000e+01,1.100000e+01
25%,9.921470e+03,5.400000e+01,1.540000e+02
50%,2.739834e+04,1.360000e+02,4.410000e+02
75%,6.856930e+04,2.950000e+02,1.294000e+03
max,2.993196e+08,1.559155e+06,2.169220e+07


In [8]:
# Flag providers where payment is above 99th percentile
p99 = df['Tot_Mdcr_Pymt_Amt'].quantile(0.99)
df['is_high_cost_outlier'] = df['Tot_Mdcr_Pymt_Amt'] > p99
print(f'99th percentile payment: ${p99:,.2f}')
print(f'High cost outliers: {df["is_high_cost_outlier"].sum()}')

99th percentile payment: $1,033,737.65
High cost outliers: 12594


In [9]:
print(df.shape)
print(df.dtypes)
df.head(3)

(1259343, 14)
Rndrng_NPI                          int64
Rndrng_Prvdr_Last_Org_Name         object
Rndrng_Prvdr_State_Abrvtn          object
Rndrng_Prvdr_Type                  object
Tot_Benes                           int64
Tot_Srvcs                         float64
Tot_Sbmtd_Chrg                    float64
Tot_Mdcr_Alowd_Amt                float64
Tot_Mdcr_Pymt_Amt                 float64
Bene_CC_PH_CKD_V2_Pct             float64
Bene_CC_PH_Diabetes_V2_Pct        float64
Bene_CC_PH_HF_NonIHD_V2_Pct       float64
Bene_CC_PH_Hypertension_V2_Pct    float64
is_high_cost_outlier                 bool
dtype: object


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,Tot_Benes,Tot_Srvcs,Tot_Sbmtd_Chrg,Tot_Mdcr_Alowd_Amt,Tot_Mdcr_Pymt_Amt,Bene_CC_PH_CKD_V2_Pct,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,is_high_cost_outlier
0,1003000126,Enkeshafi,MD,Hospitalist,344,814.0,173087.77,78590.79,62198.36,50.0,45.0,53.0,75.0,False
1,1003000134,Cibull,IL,Pathology,2093,4839.0,726858.00,171256.75,127984.95,20.0,22.0,13.0,63.0,False
2,1003000142,Khalil,OH,Anesthesiology,325,1455.0,451425.00,155879.22,118122.12,28.0,44.0,24.0,75.0,False


In [10]:
df['Tot_Srvcs'] = df['Tot_Srvcs'].astype(int)
print(df['Tot_Srvcs'].dtype)

int64


In [13]:
# Top 10 specialties by average Medicare payment
top_specialties = df.groupby('Rndrng_Prvdr_Type')['Tot_Mdcr_Pymt_Amt']\
    .mean()\
    .sort_values(ascending=False)\
    .head(10)
print(top_specialties)

Rndrng_Prvdr_Type
Clinical Laboratory                  2.338831e+06
All Other Suppliers                  1.645066e+06
Radiation Therapy Center             1.080372e+06
Ambulatory Surgical Center           9.517303e+05
Hematology-Oncology                  7.185028e+05
Portable X-Ray Supplier              7.174986e+05
Micrographic Dermatologic Surgery    6.334473e+05
Medical Oncology                     5.110346e+05
Rheumatology                         4.879588e+05
Ambulance Service Provider           4.696362e+05
Name: Tot_Mdcr_Pymt_Amt, dtype: float64


In [12]:
top_specialties.map(lambda x: f"${x:,.2f}")

Rndrng_Prvdr_Type
Clinical Laboratory                  $2,338,831.16
All Other Suppliers                  $1,645,066.48
Radiation Therapy Center             $1,080,372.09
Ambulatory Surgical Center             $951,730.29
Hematology-Oncology                    $718,502.78
Portable X-Ray Supplier                $717,498.58
Micrographic Dermatologic Surgery      $633,447.27
Medical Oncology                       $511,034.58
Rheumatology                           $487,958.84
Ambulance Service Provider             $469,636.23
Name: Tot_Mdcr_Pymt_Amt, dtype: object

In [15]:
df.to_csv(r'C:\Docs\AWS Certification\cms-claims-analytics-pipeline\data\cms_provider_cleaned.csv', index=False)
print('Saved!')

Saved!
